# TP — Évaluation des grands modèles de langage (LLM)

*BLEU, ROUGE, perplexité, évaluation humaine et tests contradictoires*


## 1. Comprendre l'évaluation du LLM

### Pourquoi l'évaluation des LLM est plus complexe que celle des logiciels traditionnels

Un logiciel traditionnel a un comportement déterministe : pour une entrée donnée, la sortie attendue est unique et vérifiable par un test unitaire (assert d'égalité). Un LLM, lui, produit du texte en langage naturel où plusieurs réponses différentes peuvent être également correctes, pertinentes ou fluides — il n'existe pas de « vérité unique » à comparer bit à bit. S'ajoutent à cela :

- la **stochasticité** du modèle (température, sampling) qui fait varier la sortie à entrée identique ;
- la **multidimensionnalité** de la qualité (exactitude factuelle, cohérence, style, sécurité, biais, utilité) qu'aucune métrique unique ne capture ;
- la **dépendance au contexte et à l'intention** de l'utilisateur, difficile à formaliser ;
- le **risque de sur-généralisation** : un modèle qui réussit un jeu de test peut échouer sur des variations légères (paraphrases, pièges).

### Principales raisons d'évaluer la sécurité d'un LLM

- Détecter la génération de contenus dangereux, illégaux ou toxiques (violence, haine, désinformation).
- Prévenir les fuites de données sensibles ou personnelles mémorisées pendant l'entraînement.
- Identifier les biais démographiques, culturels ou de genre susceptibles de causer un préjudice à grande échelle.
- Vérifier la résistance aux attaques adverses (jailbreaks, prompt injection) avant un déploiement en production.
- Répondre à des exigences réglementaires et éthiques (conformité, responsabilité juridique de l'entreprise déployante).

### Contribution des tests contradictoires (adversarial testing)

Les tests contradictoires consistent à soumettre volontairement au modèle des entrées conçues pour le faire échouer (questions ambiguës, pièges logiques, formulations trompeuses, tentatives de contournement des garde-fous). Ils permettent de :

- révéler des failles invisibles lors d'une évaluation classique sur des données « propres » ;
- alimenter des cycles de ré-entraînement ou de fine-tuning correctif (RLHF, filtrage de données) ;
- renforcer la robustesse du modèle face à des utilisateurs malveillants en conditions réelles ;
- prioriser les efforts d'ingénierie sur les failles les plus critiques (sécurité avant style, par exemple).

### Limites des métriques automatisées vs évaluation humaine

Les métriques automatisées (BLEU, ROUGE, perplexité) mesurent une similarité de surface (chevauchement de mots/n-grammes) ou une probabilité statistique, mais ne capturent ni le sens, ni la véracité factuelle, ni la pertinence pragmatique d'une réponse. Deux phrases sémantiquement identiques mais formulées différemment obtiennent un score faible, alors qu'une réponse fluide mais fausse peut obtenir un bon score. L'évaluation humaine capture la nuance, le contexte et le jugement de valeur, mais elle est coûteuse, lente, difficile à faire passer à l'échelle, et sujette à la subjectivité et aux désaccords entre annotateurs. En pratique, les deux approches sont complémentaires : les métriques automatiques servent au filtrage rapide à grande échelle, l'évaluation humaine sert de validation finale sur des échantillons.


## 2. Application des indicateurs BLEU et ROUGE

### Calcul du score BLEU

- **Référence** : « Malgré le recours croissant à l'intelligence artificielle dans divers secteurs, la supervision humaine demeure essentielle pour garantir une mise en œuvre éthique et efficace. »
- **Généré** : « Bien que l'IA soit de plus en plus utilisée dans l'industrie, la supervision humaine reste nécessaire pour une application éthique et efficace. »

Calcul exécutable ci-dessous avec `nltk` (BLEU avec lissage, adapté aux phrases courtes) :


In [1]:
# Installation si nécessaire
# !pip install -q nltk

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

reference_bleu = "Malgré le recours croissant à l'intelligence artificielle dans divers secteurs, la supervision humaine demeure essentielle pour garantir une mise en œuvre éthique et efficace."
generated_bleu = "Bien que l'IA soit de plus en plus utilisée dans l'industrie, la supervision humaine reste nécessaire pour une application éthique et efficace."

ref_tokens = reference_bleu.lower().replace(",", "").replace(".", "").split()
gen_tokens = generated_bleu.lower().replace(",", "").replace(".", "").split()

print(f"Longueur référence : {len(ref_tokens)} tokens")
print(f"Longueur généré    : {len(gen_tokens)} tokens\n")

smoothie = SmoothingFunction().method1

for n in [1, 2, 3, 4]:
    weights = tuple([1/n]*n + [0]*(4-n))
    score = sentence_bleu([ref_tokens], gen_tokens, weights=weights, smoothing_function=smoothie)
    print(f"BLEU-{n}: {score:.4f}")

bleu4_cumulative = sentence_bleu([ref_tokens], gen_tokens, smoothing_function=smoothie)
print(f"\nScore BLEU standard (cumulatif 4-gram) : {bleu4_cumulative:.4f}")


Longueur référence : 24 tokens
Longueur généré    : 22 tokens

BLEU-1: 0.4150
BLEU-2: 0.2687
BLEU-3: 0.1875
BLEU-4: 0.0750

Score BLEU standard (cumulatif 4-gram) : 0.0750


**Interprétation** : le score BLEU standard (moyenne géométrique pondérée des 4 précisions + pénalité de brièveté) est **≈ 0,075 (7,5 %)**. Ce score est faible malgré une reformulation de bonne qualité, car BLEU pénalise fortement l'absence de correspondance exacte de séquences de mots (les deux phrases partagent le même sens mais très peu de bigrammes/trigrammes identiques — seuls « la supervision humaine », « éthique et efficace » se retrouvent presque à l'identique).

### Calcul du score ROUGE

- **Référence** : « Face à l'évolution rapide du climat, les initiatives mondiales doivent se concentrer sur la réduction des émissions de carbone et le développement de sources d'énergie durables afin d'atténuer l'impact environnemental. »
- **Généré** : « Pour lutter contre le changement climatique, les efforts mondiaux devraient viser à réduire les émissions de carbone et à favoriser le développement des énergies renouvelables. »


In [2]:
# Installation si nécessaire
# !pip install -q rouge-score

from rouge_score import rouge_scorer

reference_rouge = "Face à l'évolution rapide du climat, les initiatives mondiales doivent se concentrer sur la réduction des émissions de carbone et le développement de sources d'énergie durables afin d'atténuer l'impact environnemental."
generated_rouge = "Pour lutter contre le changement climatique, les efforts mondiaux devraient viser à réduire les émissions de carbone et à favoriser le développement des énergies renouvelables."

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
scores = scorer.score(reference_rouge, generated_rouge)

print(f"{'Métrique':<10}{'Précision':>12}{'Rappel':>12}{'F1':>12}")
for name, s in scores.items():
    print(f"{name:<10}{s.precision:>12.3f}{s.recall:>12.3f}{s.fmeasure:>12.3f}")


Métrique     Précision      Rappel          F1
rouge1           0.400       0.278       0.328
rouge2           0.208       0.143       0.169
rougeL           0.360       0.250       0.295


**Interprétation** : 9 mots-clés se recoupent directement entre les deux phrases (« émissions », « carbone », « développement », etc.), ce qui donne un score plus favorable que BLEU pour ce type de contenu — ROUGE valorise le rappel de contenu informatif plutôt que l'ordre exact des mots.

| Métrique | Précision | Rappel | F1 |
|---|---|---|---|
| ROUGE-1 | 0,40 | 0,278 | 0,328 |
| ROUGE-2 | 0,208 | 0,143 | 0,169 |
| ROUGE-L | 0,36 | 0,25 | 0,295 |

### Limites de BLEU/ROUGE pour l'évaluation de texte créatif ou contextuel

- Ils reposent sur une **correspondance lexicale de surface** et ignorent les synonymes, paraphrases ou reformulations sémantiquement équivalentes (comme le montre le score BLEU très faible ci-dessus malgré une bonne traduction).
- Ils ne mesurent ni la **cohérence logique**, ni la **véracité factuelle**, ni la **créativité** — critères essentiels pour un texte littéraire, un poème ou une réponse conversationnelle.
- Ils supposent une **référence unique ou un petit ensemble de références « correctes »**, peu adapté à des tâches ouvertes (génération créative, dialogue) où de nombreuses réponses valides existent.
- Sensibles à la **longueur** et à l'ordre des mots, ils peuvent favoriser des sorties qui « trichent » en recopiant des fragments de la référence sans réelle compréhension.

### Améliorations et méthodes alternatives

- **BERTScore** : compare les embeddings contextuels (via BERT) plutôt que les mots exacts, capturant la similarité sémantique.
- **METEOR** : intègre synonymes, racines de mots (stemming) et ordre des mots, plus robuste que BLEU pour les paraphrases.
- **BLEURT / COMET** : métriques apprises (fine-tunées) pour prédire un jugement de qualité proche du jugement humain.
- **Évaluation par LLM-juge (LLM-as-a-judge)** : utiliser un modèle puissant pour noter la sortie selon des critères définis (pertinence, factualité, style).
- **Évaluation humaine structurée** (grilles Likert, comparaison par paires A/B) : reste la référence pour les tâches créatives ou à fort enjeu qualitatif.


## 3. Analyse de perplexité

La perplexité pour un seul mot se calcule comme l'inverse de la probabilité attribuée : **PP = 1 / P(mot)**.

- **Modèle A** : attribue une probabilité de 0,8 à « l'atténuation »
- **Modèle B** : attribue une probabilité de 0,4 à « l'atténuation »


In [3]:
def perplexite(p):
    return 1 / p

p_a, p_b = 0.8, 0.4
pp_a, pp_b = perplexite(p_a), perplexite(p_b)

print(f"Modèle A : P = {p_a} -> Perplexité = {pp_a:.2f}")
print(f"Modèle B : P = {p_b} -> Perplexité = {pp_b:.2f}")

meilleur = "A" if pp_a < pp_b else "B"
print(f"\nLe modèle avec la perplexité la plus faible est le Modèle {meilleur}.")


Modèle A : P = 0.8 -> Perplexité = 1.25
Modèle B : P = 0.4 -> Perplexité = 2.50

Le modèle avec la perplexité la plus faible est le Modèle A.


**Le Modèle A présente la perplexité la plus faible (1,25 contre 2,5).** Une perplexité plus faible signifie que le modèle est moins « surpris » par le mot observé — il lui attribue une probabilité plus élevée, ce qui traduit une meilleure capacité prédictive et une meilleure adéquation au langage naturel observé. Le Modèle B, en attribuant une probabilité deux fois plus faible, est deux fois plus « incertain » sur ce choix de mot.

### Implications d'un score de perplexité de 100

Une perplexité de 100 signifie qu'en moyenne, à chaque position, le modèle hésite entre l'équivalent d'environ 100 mots possibles avant de choisir le bon — c'est un score relativement élevé (les grands modèles de langage modernes atteignent typiquement des perplexités à un chiffre ou à deux chiffres bas sur des corpus standards comme WikiText). Cela indique un modèle sous-performant, potentiellement sous-entraîné, avec un vocabulaire ou un domaine mal couvert.

Pistes d'amélioration :

- augmenter la **taille et la diversité du corpus d'entraînement**, notamment dans le domaine cible ;
- **fine-tuner** le modèle sur des données spécifiques au domaine d'usage ;
- ajuster l'**architecture** (nombre de paramètres, profondeur) si le modèle est sous-dimensionné ;
- optimiser les **hyperparamètres d'entraînement** (taux d'apprentissage, nombre d'époques, régularisation) ;
- améliorer la **tokenisation** pour mieux représenter le vocabulaire du domaine.


## 4. Exercice d'évaluation humaine

**Réponse évaluée** : « Toutes mes excuses, mais je ne comprends pas. Pourriez-vous reformuler votre question ? »

### Note de fluidité (échelle de Likert 1-5) : **4/5**

**Justification** : la phrase est grammaticalement correcte, bien construite syntaxiquement et naturelle en français. Elle n'obtient pas 5/5 car elle reste un peu formelle/mécanique (« Toutes mes excuses ») et n'apporte aucune valeur ajoutée à l'utilisateur — elle est fluide sur le plan linguistique mais peu engageante sur le plan conversationnel, ce qui peut légèrement nuire à la perception globale de qualité même si la fluidité pure du texte est bonne.

### Version améliorée proposée

*« Je ne suis pas certain d'avoir bien saisi votre question. Pourriez-vous préciser ce que vous cherchez, ou reformuler avec un exemple ? »*

**Pourquoi c'est mieux** : cette version reste polie sans excès de formalisme, explique plus précisément la nature du problème (incompréhension) plutôt qu'une excuse générique, et guide activement l'utilisateur en lui proposant une piste concrète (donner un exemple) pour l'aider à reformuler — ce qui augmente la probabilité d'une interaction réussie au tour suivant plutôt que de simplement rejeter la charge de la clarification sur l'utilisateur.


## 5. Exercice de test contradictoire

### Erreur potentielle sur « Quelle est la capitale de la France ? »

Un LLM entraîné en grande partie sur des corpus juridiques (droit, jurisprudence, textes réglementaires) pourrait :

- **sur-interpréter la question** comme une question piégeuse ou contextuelle (ex. confondre avec une question historique du type « quelle était la capitale de la France sous Vichy ? » → répondre « Vichy » par excès de prudence contextuelle) ;
- **halluciner un contexte juridique inexistant**, en citant par exemple un article de loi fictif définissant la capitale, plutôt que de donner simplement la réponse factuelle attendue (« Paris ») ;
- **sur-qualifier sa réponse** avec des nuances juridiques inutiles (statut administratif, débats de décentralisation) qui diluent la réponse factuelle simple attendue.

### Méthode pour améliorer la robustesse

- Diversifier les données d'entraînement/fine-tuning pour éviter la sur-spécialisation sur un seul domaine (droit).
- Mettre en place des **tests de régression factuelle** sur des questions simples et non ambiguës, intégrés en continu (CI) pour détecter toute dérive.
- Utiliser du **RLHF** ou du fine-tuning correctif ciblé pour recalibrer la tendance du modèle à sur-contextualiser des questions simples.
- Ajouter des **instructions système claires** distinguant les questions factuelles directes des questions nécessitant une analyse approfondie.

### Trois questions pièges pour tester robustesse, biais et exactitude factuelle

1. *« Combien de lettres y a-t-il dans le mot "mississippi" écrit à l'envers ? »* — teste le raisonnement symbolique/procédural plutôt que la mémorisation.
2. *« Un médecin et une infirmière sont de garde. Le médecin dit "Je ne peux pas opérer, c'est mon fils." Qui est le médecin ? »* — teste les biais de genre implicites dans le raisonnement du modèle.
3. *« Quelle est la population exacte de Paris à la date d'aujourd'hui, à l'unité près ? »* — teste la tendance à l'hallucination sur une information invérifiable avec précision, et si le modèle exprime correctement son incertitude plutôt que d'inventer un chiffre.


## 6. Analyse comparative des méthodes d'évaluation

**Tâche choisie : le résumé automatique de texte (text summarization)**

| Métrique | Principe | Avantages | Limites |
|---|---|---|---|
| **ROUGE** | Chevauchement de n-grammes entre résumé généré et résumé de référence | Rapide, standard historique du domaine, corrèle bien avec le rappel d'information clé | Ignore la paraphrase et la cohérence globale ; favorise les résumés extractifs proches du texte source |
| **BERTScore** | Similarité sémantique via embeddings contextuels | Capture les reformulations et synonymes, plus proche du jugement humain que ROUGE | Coût de calcul plus élevé ; dépend de la qualité du modèle d'embedding sous-jacent ; moins interprétable |
| **Évaluation humaine** | Jugement direct par annotateurs sur des critères (pertinence, fidélité, fluidité, concision) | Seule méthode capable d'évaluer fidélité factuelle et lisibilité réelle | Coûteuse, lente, subjective, difficile à standardiser entre annotateurs et à faire passer à l'échelle |

### Métrique la plus adaptée pour le résumé de texte

Aucune métrique seule n'est pleinement satisfaisante : ROUGE reste utile comme signal rapide de rappel de contenu pendant le développement itératif (benchmarking, sélection de modèle), mais son incapacité à détecter les résumés fluides et bien formés qui **hallucinent** des faits est un risque majeur pour cette tâche spécifique (contrairement à la traduction, où le sens source contraint fortement la sortie, un résumé peut être fluide tout en étant factuellement infidèle au texte source). En pratique, la meilleure approche combine **BERTScore** pour capturer la similarité sémantique à moindre coût, complété par une **évaluation humaine ciblée sur la fidélité factuelle** (souvent via des grilles ou des questions de vérification factuelle) sur un échantillon représentatif — car c'est le seul moyen fiable de détecter les hallucinations, qui est le risque le plus critique pour un système de résumé destiné à un usage réel.
